In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
import pickle
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
import import_ipynb
import model
from model import LightGestureNet
from model import InvertedResidual

In [ ]:
class GestureDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = torch.FloatTensor(X).unsqueeze(1)
        self.y = torch.LongTensor(y)
        self.transform = transform

    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        image = self.X[idx]
        label = self.y[idx]

        if self.transform:
            image = self.transform(image)

        return image, label
    
train_transform = transforms.Compose([
    transforms.RandomRotation(30),
    transforms.RandomAffine(0, scale=(0.8, 1.2)),
    transforms.RandomAffine(0, translate=(0.2, 0.2))
])

In [ ]:
with open('../dataset/train.pkl', 'rb') as f:
    X_train, y_train = pickle.load(f)
with open('../dataset/test.pkl', 'rb') as f:
    X_val, y_val = pickle.load(f)

train_dataset = GestureDataset(X_train, y_train, transform=train_transform)
val_dataset = GestureDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print("TrainDataset Length: ", len(train_dataset))
print("ValDataset Length: ", len(val_dataset))

In [ ]:
import torch.nn.init as init

def weight_init(m):
    if isinstance(m, nn.Conv2d):
        init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d):
        init.constant_(m.weight, 1)
        init.constant_(m.bias, 0)
    elif isinstance(m, nn.Linear):
        init.xavier_normal_(m.weight)
        if m.bias is not None:
            init.constant_(m.bias, 0)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = LightGestureNet().do(device)

num_epochs = 50
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model.apply(weight_init)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

best_val_loss = float('inf')
patience = 5
patience_counter = 0

In [ ]:
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()


    epoch_train_loss = train_loss / len(train_loader)
    epoch_train_acc = 100. * train_correct / train_total

    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    epoch_val_loss = val_loss / len(val_loader)
    epoch_val_acc = 100. * val_correct / val_total

    scheduler.step()

    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)

    print(f'\nEpoch {epoch+1}/{num_epochs}:')
    print(f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%')
    print(f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}')

    ACCURACY_TRAIN_THRESHOLD = 96.61
    ACCURACY_VAL_THRESHOLD = 93.73

    if epoch_train_acc >= ACCURACY_TRAIN_THRESHOLD and epoch_val_acc >= ACCURACY_VAL_THRESHOLD:
        print(f'\nAchieve the target accuracy! Epoch {epoch+1}')
        break

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'\nEarly stopping: Validation loss did not improve for {patience} epochs')
            break

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Training Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import torch
import onnx
import onnxruntime as ort
import tensorflow as tf
import numpy as np

device = torch.device("cpu")

model.load_state_dict(
    torch.load('best_model.pth', map_location=device, weights_only=False)
)
model.eval()
model.to(device)

# Pytorch
torch.save(model.state_dict(), 'gesture_model.pth')

# ONNX
dummy_input = torch.randn(1, 1, 96, 96).to(device)
onnx_model_path = 'gesture_model.onnx'
torch.onnx.export(
    model,
    dummy_input,
    onnx_model_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

# ONNX Runtime
try:
    ort_session = ort.InferenceSession(onnx_model_path)
    outputs = ort_session.run(
        None,
        {
            'input': dummy_input.numpy()
        }
    )
    print("ONNX model inference test succeeded, result:", outputs)
except Exception as e:
    print("Error during ONNX model inference: ", e)
    exit(1)


# Tensorflow SavedModel
class SimpleKerasModel(tf.keras.Model):
    def __init__(self):
        super(SimpleKerasModel, self).__init__()
        self.conv1 = tf.keras.layers.Conv2D(
            16,
            (3, 3),
            padding='same',
            activation='relu'
        )
        self.pool1 = tf.keras.layers.MaxPooling2D((2, 2))
        self.conv2 = tf.keras.layers.Conv2D(
            32,
            (3, 3),
            padding='same',
            activation='relu'
        )
        self.pool2 = tf.keras.layers.MaxPooling2D((2, 2))
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(128, activation='relu')
        self.fc2 = tf.keras.layers.Dense(10)

    def call(self, x):
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        return self.fc2(x)
    
try:
    tf_model = SimpleKerasModel()
    tf_input = tf.convert_to_tensor(np.random.randn(1, 96, 96, 1), dtype=tf.flaot32)
    tf_model(tf_input)
    tf.saved_model.save(tf_model, 'tf_gesture_model')
    print(f"SavedModel: tf_gesture_model")
except Exception as e:
    print("Error during manual TensorFlow model creation: ", e)
    exit(1)


# TFLite
try:
    converter = tf.lite.TFLiteConverter.from_saved_model('tf_gesture_model')

    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float32]

    tflite_model = converter.convert()

    with open('gesture_model.tflite', 'wb') as f:
        f.write(tflite_model)

    print("Model has been saved in the following formats:")
    print("- Pytorch (.pth)")
    print("- ONNX (.onnx)")
    print("- TFLite (.tflite)")
except Exception as e:
    print("Error during SavedModel to TFLite Conversion: ", e)
    exit(1)